# UAV Attack Dataset Preprocessing
**Dataset:** Live GPS Spoofing and Jamming (UAVAttackData)  
**Source:** IEEE DataPort  
**Attack Types:** Benign, GPS Jamming, GPS Spoofing  
**Output:** processed_UAV_Attack.csv

## Dataset Domain Knowledge — UAV Attack (Live GPS)

### How GPS Works on Drones
Drones use GPS to know their position, altitude, and velocity.
The flight controller reads GPS data constantly to navigate safely.
If GPS data is corrupted or faked, the drone flies to wrong locations
or loses stability completely.

### What Each Column Means

| Column | Meaning | Attack Signal |
|--------|---------|---------------|
| `lat_x, lon_x`      | GPS reported latitude/longitude | Fake coordinates = Spoofing |
| `alt_x`             | GPS reported altitude  | Wrong altitude = crash risk |
| `eph_x, epv_x`      | GPS accuracy estimates (horizontal/vertical) | High values = signal degraded |
| `jamming_indicator` | Built-in GPS jamming detector   | High value = jamming detected |
| `vel_m_s`           | Total velocity | Sudden changes = attack effect |
| `satellites_used`   | How many satellites connected | Low count = jamming |
| `vx, vy, vz`        | Velocity in 3 directions | Erratic values = under attack |

### The 2 Attack Types

**GPS Jamming** — blocking attack
- Attacker broadcasts strong radio signals on GPS frequency
- Drone's GPS receiver gets overwhelmed and loses signal
- Drone cannot determine its position
- Flight controller switches to failsafe mode or crashes

**GPS Spoofing** — deception attack
- Attacker sends fake GPS signals that look real
- Drone believes it is at a different location than reality
- Drone flies toward fake coordinates
- Much more dangerous than jamming — drone doesn't know it's being attacked

### Expected XAI Findings
- **SHAP on jamming_indicator** → key feature for GPS Jamming detection
- **SHAP on eph_x/epv_x** → accuracy drops during both attacks
- **SHAP on satellites_used** → drops significantly during jamming
- **LIME may disagree with SHAP** on which GPS feature matters most

In [1]:
import pandas as pd          # for loading and working with CSV data
import numpy as np           # for numerical operations (e.g. detecting Inf values)
import zipfile               # for reading files directly from inside the zip
from sklearn.preprocessing import LabelEncoder, StandardScaler  # for encoding labels and scaling features
import os                    # for creating folders on disk

In [2]:
# path to the zip file on laptop
zip_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV_Attack/UAVAttackData.zip"

# folder where we will save the extracted CSV files
output_dir = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV_Attack/extracted/"

# create the output folder if it does not already exist
os.makedirs(output_dir, exist_ok=True)

# the two processed CSV files we want — already cleaned by the dataset authors
files_to_extract = [
    "Live GPS Spoofing and Jamming/GPS Jamming/Processed/jamming-merged-gps-only.csv",
    "Live GPS Spoofing and Jamming/GPS Spoofing/Processed/spoofing-merged-gps-only.csv"
]

# open the zip file and extract only these two files (not the whole zip)
with zipfile.ZipFile(zip_path, 'r') as z:
    for f in files_to_extract:
        z.extract(f, output_dir)
        print(f" Extracted: {f}")

 Extracted: Live GPS Spoofing and Jamming/GPS Jamming/Processed/jamming-merged-gps-only.csv
 Extracted: Live GPS Spoofing and Jamming/GPS Spoofing/Processed/spoofing-merged-gps-only.csv


In [4]:
# define paths to the two extracted CSV files
jamming_path = output_dir + "Live GPS Spoofing and Jamming/GPS Jamming/Processed/jamming-merged-gps-only.csv"
spoofing_path = output_dir + "Live GPS Spoofing and Jamming/GPS Spoofing/Processed/spoofing-merged-gps-only.csv"

# load both files into dataframes
df_jamming = pd.read_csv(jamming_path)
df_spoofing = pd.read_csv(spoofing_path)

# check shape of each file (rows, columns)
print(f"Jamming rows: {df_jamming.shape[0]}, columns: {df_jamming.shape[1]}")
print(f"Spoofing rows: {df_spoofing.shape[0]}, columns: {df_spoofing.shape[1]}")

# check what label values exist in each file
print(f"\nJamming labels: {df_jamming['label'].unique()}") 
# unique() => pandas function that shows all different values that appear in a column, no repeats
# i.g.:  benign, benign, jamming, benign, jamming, jamming = ['benign', 'jamming']

print(f"Spoofing labels: {df_spoofing['label'].unique()}")

Jamming rows: 6445, columns: 84
Spoofing rows: 3622, columns: 84

Jamming labels: ['benign' 'malicious']
Spoofing labels: ['benign' 'malicious']


In [6]:
# rename 'malicious' to the specific attack type so we can tell them apart after merging
# without this, we would lose information about which attack type each row belongs to
df_jamming['label'] = df_jamming['label'].replace('malicious', 'GPS_Jamming')
df_spoofing['label'] = df_spoofing['label'].replace('malicious', 'GPS_Spoofing')

# merge both dataframes into one combined dataframe
df = pd.concat([df_jamming, df_spoofing], ignore_index=True)
# concat(combine both together)
# ignore_index: 0, 1, 2 ... 10066   ← continuous, no restart 

# confirm the merge worked correctly
print(f"Total rows after merge: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"\nAll labels: {df['label'].unique()}")
print(f"\nLabel counts:\n{df['label'].value_counts()}")

Total rows after merge: 10067
Total columns: 84

All labels: ['benign' 'GPS_Jamming' 'GPS_Spoofing']

Label counts:
label
benign          8109
GPS_Jamming     1460
GPS_Spoofing     498
Name: count, dtype: int64


In [23]:
# 1: 'timestamp' is a time value, not a feature useful for detection - drop it 
# 'time_utc_usec' = absolute UTC time
# 'timestamp_time_relative' = relative time since start
# 'ignore' => "If the column doesn't exist — just skip it quietly, don't crash"
df = df.drop(columns=['time_utc_usec', 'timestamp_time_relative'], errors='ignore')

# 2: check for missing value (NaN) in each column
# NaN = Not a Number - empty/missing value in a cell.it will break model training because the model can not do math on nothing
# df.isnnull() -> checks every single cell, returns True if empty, false if has value
# .sum() -> counts how many true(empty) per column. .sum() -> sum again for totoal number of true
nan_count = df.isnull().sum().sum()
print(f"Total NaN values: {nan_count}")

# 3: check for infinite values (Inf) — division by zero results that break model training
# df.select_dtypes(include=np.number) → selects only number columns (ignores text like label)
# np.isinf(...) → checks every number cell, returns True if it's infinity
# .sum().sum() → counts total infinity values across all columns
inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
print(f"Total Inf values: {inf_count}")

# 4: confirm shape after dropping time columns
print(f"Shape after dropping time columns: {df.shape}")
      


Total NaN values: 0
Total Inf values: 0
Shape after dropping time columns: (10067, 81)


In [18]:
print(df.columns.tolist())

['q[0]', 'q[1]', 'q[2]', 'q[3]', 'delta_q_reset[0]', 'delta_q_reset[1]', 'delta_q_reset[2]', 'delta_q_reset[3]', 'quat_reset_counter', 'lat_x', 'lon_x', 'alt_x', 'alt_ellipsoid_x', 'delta_alt', 'eph_x', 'epv_x', 'terrain_alt', 'lat_lon_reset_counter', 'alt_reset_counter', 'terrain_alt_valid', 'dead_reckoning', 'lat_y', 'lon_y', 'alt_y', 'alt_ellipsoid_y', 's_variance_m_s', 'c_variance_rad', 'eph_y', 'epv_y', 'hdop', 'vdop', 'noise_per_ms', 'jamming_indicator', 'vel_m_s', 'vel_n_m_s', 'vel_e_m_s', 'vel_d_m_s', 'cog_rad', 'heading_offset', 'fix_type', 'vel_ned_valid', 'satellites_used', 'ref_lat', 'ref_lon', 'x', 'y', 'z', 'delta_xy[0]', 'delta_xy[1]', 'delta_z', 'vx', 'vy', 'vz', 'z_deriv', 'delta_vxy[0]', 'delta_vxy[1]', 'delta_vz', 'ax', 'ay', 'az', 'heading_y', 'delta_heading', 'ref_alt', 'dist_bottom', 'eph', 'epv', 'evh', 'evv', 'xy_valid', 'z_valid', 'v_xy_valid', 'v_z_valid', 'xy_reset_counter', 'z_reset_counter', 'vxy_reset_counter', 'vz_reset_counter', 'heading_reset_counter', 

In [20]:
# separate features (X) and label (y)
# X = everything except the label column
# y = the label column only
X = df.drop(columns=['label'])
y = df['label']

# encode text labels into numbers so the model cna undertand tehm
# benign -> 0, GPS_Jamming -> 1, GPS_Spoofing -> 2 (order assigined automatically)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# print the mapping so we know which number = which label
# give both number and label name
# the results will be GPS_Jamming → 0   GPS_Spoofing → 1   benign -> 2
# why 'b' is latest? python use G = 71 (ascii), 'b' = 98
print("Label cncoding:")
for i, label in enumerate(le.classes_):
    print(f" {label} -> {i}")

# Scale all features to the same range using standarScaler
# this prevents features with large value from dominating smaller ones
# before scalling        after standardScaler    now the same range(! -3 to +3)
# lat_x      vx          lat_x      vx
# 36.204     0.007       0.12       0.45
# 36.198     0.013   =>  -0.08      1.23
# 36.211     0.002       0.31       -0.89
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nFeatures shape: {X_scaled.shape}")
print(f"Labels shape: {y_encoded.shape}")

Label cncoding:
 GPS_Jamming -> 0
 GPS_Spoofing -> 1
 benign -> 2

Features shape: (10067, 80)
Labels shape: (10067,)


In [22]:
# combine scaled features and encoded labels back into one dataframe
# X_scaled is a numpy array so we convert it back to dataframe with original column names
df_processed = pd.DataFrame(X_scaled, columns=X.columns)

# add the encoded label column
df_processed['label'] = y_encoded

# save to processed folder — same location as CICIDS2017 processed file
save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV_Attack/processed_UAV_Attack.csv"
df_processed.to_csv(save_path, index=False)

print(f"Saved: {save_path}")
print(f"Final shape: {df_processed.shape}")
print(f"\nLabel distribution:")
print(df_processed['label'].value_counts())

Saved: /Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV_Attack/processed_UAV_Attack.csv
Final shape: (10067, 81)

Label distribution:
label
2    8109
0    1460
1     498
Name: count, dtype: int64


In [24]:
# Split train/test 70/30
# stratify = keeps same class ratio in both train and test sets
# random_state=42 = reproducible split every run
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (7046, 80)
X_test shape: (3021, 80)
y_train shape: (7046,)
y_test shape: (3021,)


In [25]:
import os
save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV_Attack/processed/"
os.makedirs(save_path, exist_ok=True)

np.save(save_path + "X_train.npy", X_train)
np.save(save_path + "X_test.npy", X_test)
pd.DataFrame(y_train).to_csv(save_path + "y_train.csv", index=False)
pd.DataFrame(y_test).to_csv(save_path + "y_test.csv", index=False)
pd.DataFrame(X.columns).to_csv(save_path + "feature_names.csv", index=False)
pd.DataFrame(le.classes_).to_csv(save_path + "label_classes.csv", index=False)

print("Saved!")
print(f"X_train.npy: {X_train.shape}")
print(f"X_test.npy: {X_test.shape}")

Saved!
X_train.npy: (7046, 80)
X_test.npy: (3021, 80)


## Preprocessing Complete

| Item | Value |
|------|-------|
| Source | UAVAttackData — Live GPS Spoofing and Jamming |
| Original rows | 10,067 |
| Total features | 80 |
| Labels | GPS_Jamming=0, GPS_Spoofing=1, benign=2 |
| Class imbalance | Yes — benign 80.5% dominates |
| Train set | 7,046 rows |
| Test set | 3,021 rows |
| Split ratio | 70% train / 30% test |
| Output files | X_train.npy, X_test.npy, y_train.csv, y_test.csv |